In [2]:
pip install bertopic sentence-transformers umap-learn hdbscan nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 4.8 MB/s eta 0:00:00


In [4]:
pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.9/635.9 kB 5.4 MB/s eta 0:00:00


In [5]:


import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings("ignore")

# ── 1. INSTALL DEPENDENCIES (uncomment if needed) ─────────────────────────────
# import subprocess, sys
# subprocess.check_call([sys.executable, "-m", "pip", "install",
#     "bertopic", "sentence-transformers", "umap-learn", "hdbscan"])

# ── 2. LOAD DATA ───────────────────────────────────────────────────────────────
print("Loading data...")
df = pd.read_csv("/content/combined_reddit_data.csv")
print(f"Loaded {len(df)} rows | Columns: {df.columns.tolist()}")

# ── 3. TEXT CLEANING ───────────────────────────────────────────────────────────
import nltk
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

STOP_WORDS = set(stopwords.words("english"))

# Extra filler/connector words that add noise in topic modeling
EXTRA_FILLERS = {
    "also", "nor", "first", "second", "third", "one", "two", "three",
    "even", "still", "would", "could", "should", "may", "might", "shall",
    "will", "just", "like", "get", "got", "let", "much", "many", "really",
    "very", "quite", "rather", "though", "although", "however", "therefore",
    "thus", "hence", "furthermore", "moreover", "nevertheless", "nonetheless",
    "meanwhile", "indeed", "yet", "already", "always", "never", "often",
    "usually", "sometimes", "perhaps", "maybe", "actually", "basically",
    "literally", "honestly", "thing", "things", "something", "anything",
    "everything", "nothing", "someone", "anyone", "everyone", "way", "ways",
    "make", "made", "makes", "making", "know", "known", "knew", "think",
    "thought", "said", "say", "says", "see", "seen", "saw", "come", "came",
    "use", "used", "uses", "using", "keep", "kept", "take", "took", "taken",
    "give", "gave", "given", "look", "looked", "looks", "need", "needs",
    "go", "going", "gone", "went", "want", "wanted", "wants", "put", "puts",
    "try", "tried", "tries", "seem", "seemed", "seems", "feel", "felt",
    "feels", "become", "became", "becomes", "show", "showed", "shown",
    "means", "mean", "meant", "done", "doing", "does", "did", "lot", "lots",
    "new", "good", "bad", "big", "small", "old", "long", "right", "left",
    "well", "back", "ever", "around", "without", "within", "across", "iit",
    "iitkgp", "iitb", "iitd", "iitm", "iitr", "reddit", "deleted", "removed",
    "edit", "update", "comment", "post", "thread", "op", "amp", "http",
    "https", "www", "com", "org", "net", "x200b",
}

STOP_WORDS.update(EXTRA_FILLERS)
lemmatizer = WordNetLemmatizer()

# Reddit-specific patterns
URL_RE       = re.compile(r"http\S+|www\.\S+")
MENTION_RE   = re.compile(r"u/\w+|r/\w+")
MARKDOWN_RE  = re.compile(r"[*_~`>#|\\]")
SPECIAL_RE   = re.compile(r"[^a-zA-Z\s]")
WHITESPACE_RE = re.compile(r"\s+")

def clean_text(text):
    if not isinstance(text, str) or not text.strip():
        return None
    t = text.lower()
    t = URL_RE.sub(" ", t)
    t = MENTION_RE.sub(" ", t)
    t = MARKDOWN_RE.sub(" ", t)
    # expand common contractions
    t = re.sub(r"n't", " not", t)
    t = re.sub(r"'s|'re|'ve|'ll|'d|'m", " ", t)
    t = SPECIAL_RE.sub(" ", t)
    tokens = WHITESPACE_RE.sub(" ", t).strip().split()
    tokens = [lemmatizer.lemmatize(w) for w in tokens
              if w not in STOP_WORDS and len(w) > 2]
    return " ".join(tokens) if len(tokens) >= 4 else None  # drop near-empty docs

print("Cleaning text...")
df["clean_body"] = df["body"].apply(clean_text)

# Drop removed/deleted posts and empty cleans
df = df[~df["body"].str.lower().isin(["[deleted]", "[removed]", ""])]
df = df.dropna(subset=["clean_body"]).reset_index(drop=True)
print(f"After cleaning: {len(df)} usable comments")

# ── 4. TOPIC MODELING WITH BERTopic ───────────────────────────────────────────
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

print("Loading sentence transformer model (first run downloads ~90MB)...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Tuned for short Reddit comments
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
)

hdbscan_model = HDBSCAN(
    min_cluster_size=15,      # min comments to form a topic
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)

# Vectorizer that ignores filler words at the keyword extraction stage too
vectorizer_model = CountVectorizer(
    stop_words=list(STOP_WORDS),
    min_df=3,
    ngram_range=(1, 2),       # unigrams + bigrams for richer topic labels
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    top_n_words=10,
    verbose=True,
)

print("Fitting BERTopic (this may take a few minutes)...")
docs = df["clean_body"].tolist()
topics, probs = topic_model.fit_transform(docs)

df["topic_id"] = topics

# ── 5. PRINT TOPIC SUMMARY ────────────────────────────────────────────────────
topic_info = topic_model.get_topic_info()
total_topics = len(topic_info[topic_info["Topic"] != -1])
outliers     = int(topic_info[topic_info["Topic"] == -1]["Count"].values[0]) if -1 in topic_info["Topic"].values else 0

print("\n" + "="*70)
print(f"TOTAL TOPICS FOUND : {total_topics}")
print(f"OUTLIER COMMENTS   : {outliers}  (Topic -1, no clear cluster)")
print("="*70)
print("\n--- TOPIC LIST (share this with Claude) ---\n")

for _, row in topic_info.iterrows():
    tid = row["Topic"]
    if tid == -1:
        continue
    words = topic_model.get_topic(tid)
    keyword_str = ", ".join([w for w, _ in words[:8]])
    print(f"Topic {tid:>3} | Count: {row['Count']:>5} | Keywords: {keyword_str}")

print("\n--- END OF TOPIC LIST ---\n")

# ── 6. SAVE INTERMEDIATE FILE ─────────────────────────────────────────────────
out_path = "reddit_with_topics.csv"
df[["comment_id", "parent_id", "author", "body", "clean_body",
    "score", "created_utc", "institution", "topic_id"]].to_csv(out_path, index=False)
print(f"Saved intermediate file → {out_path}")
print("Now paste the topic list above into the chat with Claude for category mapping.")

Loading data...
Loaded 6102 rows | Columns: ['comment_id', 'parent_id', 'author', 'body', 'score', 'created_utc', 'institution']
Cleaning text...
After cleaning: 3798 usable comments
Loading sentence transformer model (first run downloads ~90MB)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-04-22 14:23:50,235 - BERTopic - Embedding - Transforming documents to embeddings.


Fitting BERTopic (this may take a few minutes)...


Batches:   0%|          | 0/119 [00:00<?, ?it/s]

2026-04-22 14:25:00,802 - BERTopic - Embedding - Completed ✓
2026-04-22 14:25:00,803 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-22 14:25:41,671 - BERTopic - Dimensionality - Completed ✓
2026-04-22 14:25:41,675 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-22 14:25:41,972 - BERTopic - Cluster - Completed ✓
2026-04-22 14:25:41,984 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-22 14:25:42,325 - BERTopic - Representation - Completed ✓



TOTAL TOPICS FOUND : 129
OUTLIER COMMENTS   : 356  (Topic -1, no clear cluster)

--- TOPIC LIST (share this with Claude) ---

Topic   0 | Count:    54 | Keywords: selfish, original, blaming, victim, academic, wrong, effort, struggle
Topic   1 | Count:    51 | Keywords: reservation, general, prolly, reserved, instead, terrace, side, solution
Topic   2 | Count:    51 | Keywords: admin, grading, professor, completely, dean, freedom, loneliness, later
Topic   3 | Count:    51 | Keywords: dekh, sub, toh, life, set, easier, kar, kuch
Topic   4 | Count:    48 | Keywords: toxic, professor, toxicity, indian, management, culture, posted, leadership
Topic   5 | Count:    45 | Keywords: pressure, exam, sake, empathetic, stressful, stress, reaching, sending
Topic   6 | Count:    45 | Keywords: others, swc, administration, management, student, prevent, head, decision
Topic   7 | Count:    42 | Keywords: nhi, hoga, attitude, nobody, fighting, abhi, pressure, hoti
Topic   8 | Count:    42 | Keywords:

In [7]:
"""
Category Assignment Script — REVISED
--------------------------------------
Improved map: S7 (Irrelevant) reduced from ~38% → ~14%
by correctly rescuing topics into S2/S3/S4/S8/S9.

Input:  reddit_with_topics.csv
Output: reddit_categorised.csv
"""

import pandas as pd

# ── LOAD ──────────────────────────────────────────────────────────────────────
df = pd.read_csv("reddit_with_topics.csv")
print(f"Loaded {len(df)} rows")

# ── CATEGORY DEFINITIONS ──────────────────────────────────────────────────────
CATEGORIES = {
    "S1": "Sympathy & Support",
    "S2": "Information Sharing",
    "S3": "Speculation / Cause Discussion",
    "S4": "Accusation / Blame",
    "S5": "Mobilization / Action",
    "S6": "Indifference / Low-effort",
    "S7": "Irrelevant",
    "S8": "Personal Reflection / Mental Health",
    "S9": "Institutional / Policy Discussion",
}

# ── REVISED TOPIC → CATEGORY MAP ─────────────────────────────────────────────
TOPIC_CATEGORY_MAP = {

    # S1 — Sympathy & Support
    # condolences, emotional support, asking people to reach out / take care
    27:  "S1",   # "soul rest peace", condolences
    42:  "S1",   # condolence meeting at hostel
    46:  "S1",   # "dm me, let's meet, everything will be sorted"
    47:  "S1",   # "listen, grades will pass, please DM"
    58:  "S1",   # "may God bless", "take care of yourself"
    63:  "S1",   # "damn man, lively guy, can't believe" + share feelings on Reddit
    67:  "S1",   # personal empathy, "feels like reading my journal"
    74:  "S1",   # "start caring about friends/neighbors in hostels"
    106: "S1",   # "rest in peace", horrified reaction, hope you heal
    118: "S1",   # "respect departed soul, don't doxx"
    119: "S1",   # "nothing more important than life"

    # S2 — Information Sharing
    # factual reports, eyewitness, news quotes, death counts, event timelines
    11:  "S2",   # accident vs suicide + roof parapet facts
    22:  "S2",   # official death notice (Chandradeep Pawar)
    23:  "S2",   # "6 suicides and 1 accidental death this year"
    29:  "S2",   # bot automod posts (sub rules/info)
    31:  "S2",   # bot recruiter/rule posts
    39:  "S2",   # reporting suicide counts at specific IITs
    40:  "S2",   # IITK stories + US/UK degree question (factual discussion)
    60:  "S2",   # "what's happening in IITK, reasons should be investigated"
    64:  "S2",   # eyewitness: police/forensic near dronagiri hostel
    77:  "S2",   # student identified — year/branch confirmed
    83:  "S2",   # news article quote (Hindustan Times / director statement)
    84:  "S2",   # murder vs suicide clarification + past rumours
    91:  "S2",   # "what happened, please tell the truth"
    92:  "S2",   # counting suicides at IITK since 2023
    103: "S2",   # 2008 campus murder — acid-treated bones found (factual)
    107: "S2",   # "heard 4-5 cases, never gets publicity"
    109: "S2",   # "what's happening, explain this" + boy jumped from M block
    114: "S2",   # 2023 case recall + 2nd post-mortem report (high court)
    116: "S2",   # "not true, don't spread rumours"
    123: "S2",   # year/branch of student, KGP suicide count

    # S3 — Speculation / Cause Discussion
    # hypothesising reasons: academic pressure, caste, parenting, peer culture
    1:   "S3",   # "academics is major reason, I also struggle"
    2:   "S3",   # mental health treatment gaps, reservation debate
    4:   "S3",   # parental pressure, materialistic expectations, depression
    8:   "S3",   # "what the f did I just read" → IIT placement reality shock
    17:  "S3",   # caste/reservation as cause of suicides
    18:  "S3",   # boomer parents, izzat culture, peer pressure
    34:  "S3",   # "desh can't do anything" — systemic fatalism debate
    45:  "S3",   # academic load + peer comparison leading to burnout
    52:  "S3",   # ragging/peer group dynamics in 1st year as cause
    53:  "S3",   # tier-3 guide exploitation → context for IIT severity
    56:  "S3",   # parental FOMO, mental barriers
    59:  "S3",   # "why so many suicides at IITs, no therapy department"
    73:  "S3",   # "give admission on merit, suicides may stop"
    75:  "S3",   # JEE grind empathy deficit as contributing factor
    88:  "S3",   # "pressure of being IITian, expectations growing"
    93:  "S3",   # copycat suicide phenomenon, depression explanation
    99:  "S3",   # who's accountable for mental health — parents/teachers
    100: "S3",   # BITS Goa leniency vs stress elsewhere (comparative)
    101: "S3",   # "lonely, outperforming, hope dies"
    108: "S3",   # burnout + self-doubt after failure (not reservation)
    122: "S3",   # social media promoting suicidal thoughts — debate

    # S4 — Accusation / Blame
    # blaming specific people, admins, professors, policies, institutions
    5:   "S4",   # professors toxic, corporate boss attitude
    6:   "S4",   # sexual misconduct in lab, rape allowed on campus
    7:   "S4",   # supervisor = associate dean, conflict of interest
    9:   "S4",   # "gimmick to show they care, just for image"
    13:  "S4",   # brother not allowed to read sister's suicide note (cover-up)
    24:  "S4",   # PhD supervisors treating students like machines
    26:  "S4",   # warden/hostel dean (Pani) always been bad
    32:  "S4",   # IIT Patna corruption, "diddy" check reference
    37:  "S4",   # "heads at the top must roll"
    48:  "S4",   # "director on US/Singapore trip while students die"
    54:  "S4",   # supreme court / legal action against professor
    55:  "S4",   # director covering for department
    66:  "S4",   # "stand up against administration"
    76:  "S4",   # "pay media to suppress news, bribe police"
    80:  "S4",   # "wealthy + influential parents let rapists stay"
    85:  "S4",   # no transparency on grades + open windows (negligence)
    102: "S4",   # committee voted to revoke PhD but prof still employed
    105: "S4",   # associate profs exploitative, blocking scholars
    110: "S4",   # profs sadistic, extend deadlines for power trip
    112: "S4",   # named accusation post (Ishan Sharma, Sanjay Mittal etc.)
    115: "S4",   # "why should victim compromise"
    125: "S4",   # "Panigrahi copy-pasted last mail, someone just died"
    126: "S4",   # "shame, copy-paste formality mails from admin"

    # S5 — Mobilization / Action
    # calls to protest, petition, media outreach, systemic change demands
    36:  "S5",   # "share screenshots with online media houses"
    41:  "S5",   # "need student union to check arrogant professors"
    49:  "S5",   # "students should start protesting against management"
    117: "S5",   # "rather than locking rooftop, fix the fcked system"

    # S6 — Indifference / Low-effort
    # memes, one-word/emoji reactions, gibberish, dismissive banter
    30:  "S6",   # pure Hindi profanity, no content
    35:  "S6",   # "wtf bhai, mat shuru karo" (dismissive)
    43:  "S6",   # gif/meme reactions only
    50:  "S6",   # off-topic rant about mosquitoes/bathrooms
    62:  "S6",   # pure abuse (maa chudwale bsdk)
    68:  "S6",   # gibberish/meme text
    70:  "S6",   # "soo sad" emoji-only one-liner
    71:  "S6",   # completely off-topic
    78:  "S6",   # branch banter (MSE/ECO/CHE), no substance
    81:  "S6",   # JoSAA bot removal + "jee disqualified shit heads"
    86:  "S6",   # "skill issue" dismissive response
    90:  "S6",   # nonsense/meme comment
    94:  "S6",   # "yes bro completely agree" — zero content
    96:  "S6",   # "WTF!!!! ye kya ho raha hai" — pure reaction
    104: "S6",   # "tell me the name" / Bollywood movie ref
    113: "S6",   # "Y23, 6-7 hogye" — minimal / low effort
    121: "S6",   # "ragebait lite" — dismissive label

    # S7 — Irrelevant
    # completely unrelated: placements, JEE ranks, campus food, college comparisons
    3:   "S7",   # research survey + CG/freedom of choice tangent
    10:  "S7",   # BITS undergraduate quality, reputation debate
    16:  "S7",   # PhD admissions / mess food debate
    19:  "S7",   # ERP messages, hostel room quality, rumour room
    20:  "S7",   # IIT tag + women/relationships tangent
    21:  "S7",   # orgy parties among PhD, CPI/branch opener stats
    25:  "S7",   # school vs college friends quality
    28:  "S7",   # entrepreneurship, education loan, job market
    33:  "S7",   # admin sucks + "IIT life set" myth (generic rant)
    44:  "S7",   # JEE rank / CSE admission discussion
    51:  "S7",   # "shortcuts in IIT" general culture rant
    61:  "S7",   # professor startup inquiry, how to deal with profs
    65:  "S7",   # placement CTC / core vs non-core debate
    72:  "S7",   # BITS Pilani vs mediocre colleges ranking
    82:  "S7",   # BITS attendance policy, campus transformation
    87:  "S7",   # BITS fee too high opinion
    89:  "S7",   # attendance policy debate details
    97:  "S7",   # BITS Hyderabad campus culture
    111: "S7",   # placement/internship cheating
    120: "S7",   # "clg ppl are selfish, better alone" (peer culture rant)
    124: "S7",   # "colleges produce glorified labourers"
    127: "S7",   # CUET/NLU/NID/NIFT alternatives

    # S8 — Personal Reflection / Mental Health
    # first-person sharing of own struggles, loneliness, burnout, ideation
    14:  "S8",   # IITK PhD thankful for supervisor; personal relief
    38:  "S8",   # "came to Reddit to distract from breakdown, exam in 1hr"
    79:  "S8",   # personal advice for suicidal ideation + own shame/aversion
    98:  "S8",   # "same, my whole life went like this, fuck this world"
    101: "S8",   # "lonely, can't achieve expectations" (personal)

    # S9 — Institutional / Policy Discussion
    # analysing systemic failures, education policy, counselling infra
    0:   "S9",   # agentic AI nihilism in IIT morale + news channels
    12:  "S9",   # gymkhana infrastructure gaps, admin pressure on students
    15:  "S9",   # admin response failures, wrong priorities (exams > lives)
    57:  "S9",   # "Indian education system doomed, top places not safe"
    69:  "S9",   # Kanpur academic pressure, professors don't care policy
    95:  "S9",   # ICS renamed, counselling system inadequacy
}

# ── ASSIGN ────────────────────────────────────────────────────────────────────
df["category_code"] = df["topic_id"].map(TOPIC_CATEGORY_MAP)
df["category_label"] = df["category_code"].map(CATEGORIES)

mapped   = df["category_code"].notna().sum()
unmapped = df["category_code"].isna().sum()
print(f"Direct map:  {mapped} assigned  |  Unmapped (LLM fallback): {unmapped}")

# ── UNMAPPED → S7 ────────────────────────────────────────────────────────────
# Topic -1 outliers (no clear cluster) default to Irrelevant
df["category_code"]  = df["category_code"].fillna("S7")
df["category_label"] = df["category_label"].fillna(CATEGORIES["S7"])

# ── SUMMARY ───────────────────────────────────────────────────────────────────
print("\n=== FINAL CATEGORY DISTRIBUTION ===")
dist = df.groupby(["category_code", "category_label"]).size().reset_index(name="count")
dist = dist.sort_values("category_code")
for _, row in dist.iterrows():
    pct = row["count"] / len(df) * 100
    bar = "█" * int(pct / 1)
    print(f"  {row['category_code']} | {row['category_label']:<40} | {row['count']:>5} ({pct:4.1f}%) {bar}")

# ── SAVE ──────────────────────────────────────────────────────────────────────
out_cols = ["comment_id", "parent_id", "author", "body", "score",
            "created_utc", "institution", "topic_id",
            "category_code", "category_label"]
df[out_cols].to_csv("reddit_categorised.csv", index=False)
print(f"\nSaved → reddit_categorised.csv  ({len(df)} rows)")

Loaded 3798 rows
Direct map:  3427 assigned  |  Unmapped (LLM fallback): 371

=== FINAL CATEGORY DISTRIBUTION ===
  S1 | Sympathy & Support                       |   270 ( 7.1%) ███████
  S2 | Information Sharing                      |   489 (12.9%) ████████████
  S3 | Speculation / Cause Discussion           |   594 (15.6%) ███████████████
  S4 | Accusation / Blame                       |   634 (16.7%) ████████████████
  S5 | Mobilization / Action                    |   108 ( 2.8%) ██
  S6 | Indifference / Low-effort                |   393 (10.3%) ██████████
  S7 | Irrelevant                               |   983 (25.9%) █████████████████████████
  S8 | Personal Reflection / Mental Health      |   126 ( 3.3%) ███
  S9 | Institutional / Policy Discussion        |   201 ( 5.3%) █████

Saved → reddit_categorised.csv  (3798 rows)
